<a href="https://colab.research.google.com/github/Honorine-Kabore/genomic_mosq/blob/main/Fst.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q --no-warn-conflicts malariagen_data

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 22.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.8/215.8 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.7/71.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.9/775.9 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.3/211.3 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 M

In [ ]:
import malariagen_data
import allel
import numpy as np
import plotly.express as px
import plotly.io as pio
import pandas as pd

In [ ]:
# plotting setup
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.gridspec import GridSpec
import matplotlib_venn as venn

In [ ]:
#Mounting Google Drive
import os
from google.colab import drive
drive.mount("drive")

# make dir
results_dir = "drive/MyDrive/Genomic/"
os.makedirs(results_dir, exist_ok=True)

Mounted at drive


In [ ]:
ag3 = malariagen_data.Ag3(pre=True, results_cache=results_dir)
ag3

<MalariaGEN Ag3 API client>
Storage URL                           : gs://vo_agam_release_master_us_central1/
Data releases available               : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16, 3.17
Results cache                         : /content/drive/MyDrive/Genomic
Cohorts analysis                      : 20260120
AIM analysis                          : 20220528
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.6.0
Client location                       : Utah, United States (Google Cloud us-west3)
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16, 3.17
---
Please note that data are subject to terms of use,
for more information see https://www.malariagen.net/data
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.6.0/Ag3.html

In [ ]:
sets =['3.7', '3.8','3.9', '3.10','3.11']
df_samples = ag3.sample_metadata(sample_sets=sets)
sample_query='country=="Burkina Faso"'


In [ ]:
# Create a new cohort for taxon
bf_samples = df_samples.query('country == "Burkina Faso" and year > 2004')
cohort_ta = {}
for ta in bf_samples.taxon.unique():
 for lo in bf_samples.location.unique():
  sample_list = list(bf_samples.query("taxon==@ta").sample_id)
  cohort_ta['An.'+ta] = "sample_id in ['"+ "', '".join(sample_list) + "']"

In [ ]:
#create a new cohort with location, taxon
bf_samples = df_samples.query('country == "Burkina Faso" and year > 2004')
cohort_re = {}
for ta in bf_samples.taxon.unique():
    for re in bf_samples.admin1_name.unique():
        sample_list = list(bf_samples.query("admin1_name==@re and taxon==@ta").sample_id)
        cohort_re[re + "_" + ta[:4]] = "sample_id in ['"+ "', '".join(sample_list) + "']"

# Fst

In [ ]:
region="3L:15,000,000-41,000,000"
site_mask='gamb_colu_arab'
n_jack=200

In [ ]:
#fst chromosome arms 3L
fst_3L = ag3.pairwise_average_fst(
    region='3L',
    cohorts=cohort_ta,
    sample_query="country == 'Burkina Faso'",
    sample_sets=sets,
    n_jack=n_jack,
    #site_mask=site_mask,
)
fst_3L

Cohort (An.unassigned) has insufficient samples (2) for requested cohort size (15), dropping.


Compute SNP allele counts:   0%|          | 0/4488 [00:00<?, ?it/s]

Compute SNP allele counts:   0%|          | 0/6392 [00:00<?, ?it/s]

Compute SNP allele counts:   0%|          | 0/4488 [00:00<?, ?it/s]

Compute SNP allele counts:   0%|          | 0/4488 [00:00<?, ?it/s]

...

Compute SNP allele counts:   0%|          | 0/6392 [00:00<?, ?it/s]

Compute SNP allele counts:   0%|          | 0/952 [00:00<?, ?it/s]

Compute SNP allele counts:   0%|          | 0/4488 [00:00<?, ?it/s]

Compute SNP allele counts:   0%|          | 0/952 [00:00<?, ?it/s]

,cohort1,cohort2,fst,se
0,An.gambiae,An.coluzzii,0.040236,0.002961
1,An.gambiae,An.arabiensis,0.217466,0.008347
2,An.gambiae,An.gcx4,0.197820,0.004064
3,An.coluzzii,An.arabiensis,0.242145,0.007994
4,An.coluzzii,An.gcx4,0.190906,0.004082
5,An.arabiensis,An.gcx4,0.394904,0.006070


In [ ]:
fst_3L.to_csv('drive/MyDrive/Genomepop/fst/fst_3L.csv', index=False)

In [ ]:
ag3.plot_pairwise_average_fst(fst_3L, color_continuous_scale='blues')

In [ ]:
ag3.plot_pairwise_average_fst(fst_3L, annotation="standard error", color_continuous_scale='blues')

In [ ]:
ag3.plot_pairwise_average_fst(fst_3L, annotation="Z score", zmax=0.03, color_continuous_scale='blues')

In [ ]:
pairwise_fst_df = ag3.pairwise_average_fst(
    region=region,
     cohorts=cohort_re,
    sample_query="country == 'Burkina Faso'",
    sample_sets=sets,
    n_jack=n_jack,
    site_mask=site_mask,
    min_cohort_size=10
)
pairwise_fst_df

Cohort (Boucle du Mouhoun_gamb) has insufficient samples (2) for requested cohort size (10), dropping.
Cohort (Sahel_gamb) has insufficient samples (0) for requested cohort size (10), dropping.
Cohort (Plateau Central_gamb) has insufficient samples (0) for requested cohort size (10), dropping.
Cohort (Centre-Ouest_gamb) has insufficient samples (0) for requested cohort size (10), dropping.
Cohort (Plateau Central_colu) has insufficient samples (3) for requested cohort size (10), dropping.
Cohort (Centre-Ouest_colu) has insufficient samples (0) for requested cohort size (10), dropping.
Cohort (Cascades_arab) has insufficient samples (8) for requested cohort size (10), dropping.
Cohort (Plateau Central_arab) has insufficient samples (4) for requested cohort size (10), dropping.
Cohort (Centre-Ouest_arab) has insufficient samples (0) for requested cohort size (10), dropping.
Cohort (Hauts-Bassins_unas) has insufficient samples (0) for requested cohort size (10), dropping.
Cohort (Boucle d

Compute SNP allele counts:   0%|          | 0/4473 [00:00<?, ?it/s]

Compute SNP allele counts:   0%|          | 0/1033 [00:00<?, ?it/s]

Compute SNP allele counts:   0%|          | 0/1033 [00:00<?, ?it/s]

Compute SNP allele counts:   0%|          | 0/2753 [00:00<?, ?it/s]

...

Compute SNP allele counts:   0%|          | 0/1033 [00:00<?, ?it/s]

Compute SNP allele counts:   0%|          | 0/2409 [00:00<?, ?it/s]

Compute SNP allele counts:   0%|          | 0/1033 [00:00<?, ?it/s]

Compute SNP allele counts:   0%|          | 0/1033 [00:00<?, ?it/s]

,cohort1,cohort2,fst,se
0,Hauts-Bassins_gamb,Cascades_gamb,0.000153,0.000047
1,Hauts-Bassins_gamb,Centre-Sud_gamb,0.000141,0.000079
2,Hauts-Bassins_gamb,Est_gamb,0.000260,0.000049
3,Hauts-Bassins_gamb,Hauts-Bassins_colu,0.023142,0.001321
4,Hauts-Bassins_gamb,Boucle du Mouhoun_colu,0.022996,0.001312
...,...,...,...,...
115,Centre-Sud_arab,Sahel_arab,0.000156,0.000351
116,Centre-Sud_arab,Cascades_gcx4,0.410119,0.006030
117,Est_arab,Sahel_arab,0.000962,0.000273
118,Est_arab,Cascades_gcx4,0.402845,0.005941


In [ ]:
import seaborn as sns

In [ ]:
pairwise_fst_df.to_csv('drive/MyDrive/Genomepop/fst/fst_3L_region')

In [ ]:
fst=ag3.plot_pairwise_average_fst(pairwise_fst_df)
fst

In [ ]:
ag3.plot_pairwise_average_fst(pairwise_fst_df, annotation=None, zmax=0.03,
                              color_continuous_scale='greens')